In [2]:
import pandas as pd
from pathlib import Path
from datetime import datetime



In [2]:
fact = pd.read_parquet(
    "../data/interim/fact_validations.parquet"
)

print("Fact table shape:", fact.shape)

Fact table shape: (79149, 37)


In [3]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print("Processed folder ready:", processed_dir)

Processed folder ready: ..\data\processed


In [4]:
agg_ridership_by_month = (
    fact.groupby("year_month")
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        unique_cards=("card_id", "nunique"),
        avg_daily_boardings=("passenger_count", "mean")
    )
    .reset_index()
)

agg_ridership_by_month["total_revenue"] = (
    agg_ridership_by_month["total_revenue"].round(2)
)

agg_ridership_by_month["avg_daily_boardings"] = (
    agg_ridership_by_month["avg_daily_boardings"].round(4)
)

agg_ridership_by_month = agg_ridership_by_month.sort_values(
    "year_month"
)

print(agg_ridership_by_month)

   year_month  total_boardings  total_revenue  unique_cards  \
0     2024-01          14000.0       25793.45          9174   
1     2024-02          13128.0       24297.69          8757   
2     2024-03          14022.0       25706.87          9161   
3     2024-04          13414.0       24606.95          8975   
4     2024-05          14088.0       25664.91          9223   
5     2024-06          13365.0       24573.98          8860   
6     2024-07            559.0        1013.42           451   
7     2024-08            569.0        1003.09           470   
8     2024-09            533.0         996.67           434   
9     2024-10            504.0         905.73           415   
10    2024-11            546.0         996.07           446   
11    2024-12            519.0         952.92           419   

    avg_daily_boardings  
0                1.0757  
1                1.0806  
2                1.0805  
3                1.0716  
4                1.0766  
5                1.0787 

In [5]:
daily_boardings = (
    fact.groupby(["year_month", "date"])
    .agg(
        daily_boardings=("passenger_count", "sum")
    )
    .reset_index()
)

avg_daily = (
    daily_boardings.groupby("year_month")
    .agg(
        avg_daily_boardings=("daily_boardings", "mean")
    )
    .reset_index()
)

avg_daily["avg_daily_boardings"] = (
    avg_daily["avg_daily_boardings"].round(4)
)

print(avg_daily)

   year_month  avg_daily_boardings
0     2024-01             451.6129
1     2024-02             452.6897
2     2024-03             452.3226
3     2024-04             447.1333
4     2024-05             454.4516
5     2024-06             445.5000
6     2024-07              93.1667
7     2024-08              94.8333
8     2024-09              88.8333
9     2024-10              84.0000
10    2024-11              91.0000
11    2024-12              86.5000


In [6]:
agg_ridership_by_month = agg_ridership_by_month.drop(
    columns=["avg_daily_boardings"]
)

agg_ridership_by_month = agg_ridership_by_month.merge(
    avg_daily,
    on="year_month",
    how="left",
    validate="one_to_one"
)

agg_ridership_by_month["total_revenue"] = (
    agg_ridership_by_month["total_revenue"].round(2)
)

agg_ridership_by_month["avg_daily_boardings"] = (
    agg_ridership_by_month["avg_daily_boardings"].round(4)
)

agg_ridership_by_month = agg_ridership_by_month.sort_values(
    "year_month"
)

print(agg_ridership_by_month)

   year_month  total_boardings  total_revenue  unique_cards  \
0     2024-01          14000.0       25793.45          9174   
1     2024-02          13128.0       24297.69          8757   
2     2024-03          14022.0       25706.87          9161   
3     2024-04          13414.0       24606.95          8975   
4     2024-05          14088.0       25664.91          9223   
5     2024-06          13365.0       24573.98          8860   
6     2024-07            559.0        1013.42           451   
7     2024-08            569.0        1003.09           470   
8     2024-09            533.0         996.67           434   
9     2024-10            504.0         905.73           415   
10    2024-11            546.0         996.07           446   
11    2024-12            519.0         952.92           419   

    avg_daily_boardings  
0              451.6129  
1              452.6897  
2              452.3226  
3              447.1333  
4              454.4516  
5              445.5000 

In [7]:
output_path = processed_dir / "agg_ridership_by_month.csv"

agg_ridership_by_month.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_ridership_by_month))
print("Columns:", len(agg_ridership_by_month.columns))

Saved: ..\data\processed\agg_ridership_by_month.csv
Rows: 12
Columns: 5


In [8]:
fact["day_type"] = fact["is_weekend"].map({
    False: "weekday",
    True: "weekend"
})

print(fact[["hour", "is_weekend", "day_type"]].head())

   hour  is_weekend day_type
0    18       False  weekday
1     8       False  weekday
2    13       False  weekday
3    17        True  weekend
4    18       False  weekday


In [9]:
hourly_base = (
    fact.groupby(["hour", "day_type"])
    .agg(
        total_boardings=("passenger_count", "sum"),
        avg_boardings_per_day=("passenger_count", "mean")
    )
    .reset_index()
)

hourly_base["avg_boardings_per_day"] = (
    hourly_base["avg_boardings_per_day"].round(4)
)

print(hourly_base.head(10))

   hour day_type  total_boardings  avg_boardings_per_day
0     0  weekday            155.0                 1.0544
1     0  weekend             43.0                 1.1026
2     1  weekday            124.0                 1.0508
3     1  weekend             42.0                 1.0500
4     2  weekday            163.0                 1.0795
5     2  weekend             31.0                 1.0333
6     3  weekday            117.0                 1.0446
7     3  weekend             68.0                 1.0794
8     4  weekday            167.0                 1.0844
9     4  weekend             62.0                 1.1071


In [12]:
hourly_base["daily_total"] = (
    hourly_base.groupby("day_type")["total_boardings"]
    .transform("sum")
)

hourly_base["pct_of_daily_total"] = (
    hourly_base["total_boardings"] / hourly_base["daily_total"]
).round(4)

agg_ridership_by_hour = (
    hourly_base
    .drop(columns="daily_total")
    .sort_values(["day_type", "hour"])
    .reset_index(drop=True)
)

print(agg_ridership_by_hour.head(10))

   hour day_type  total_boardings  avg_boardings_per_day  pct_of_daily_total
0     0  weekday            155.0                 1.0544              0.0025
1     1  weekday            124.0                 1.0508              0.0020
2     2  weekday            163.0                 1.0795              0.0027
3     3  weekday            117.0                 1.0446              0.0019
4     4  weekday            167.0                 1.0844              0.0027
5     5  weekday            388.0                 1.0689              0.0064
6     6  weekday           1575.0                 1.0714              0.0258
7     7  weekday           4830.0                 1.0743              0.0792
8     8  weekday           7376.0                 1.0765              0.1210
9     9  weekday           5495.0                 1.0743              0.0901


In [14]:
hourly_base["daily_total"] = (
    hourly_base.groupby("day_type")["total_boardings"]
    .transform("sum")
)

hourly_base["pct_of_daily_total"] = (
    hourly_base["total_boardings"] / hourly_base["daily_total"]
).round(4)

agg_ridership_by_hour = (
    hourly_base
    .drop(columns="daily_total")
    .sort_values(["day_type", "hour"])
    .reset_index(drop=True)
)

print(agg_ridership_by_hour.head(10))

   hour day_type  total_boardings  avg_boardings_per_day  pct_of_daily_total
0     0  weekday            155.0                 1.0544              0.0025
1     1  weekday            124.0                 1.0508              0.0020
2     2  weekday            163.0                 1.0795              0.0027
3     3  weekday            117.0                 1.0446              0.0019
4     4  weekday            167.0                 1.0844              0.0027
5     5  weekday            388.0                 1.0689              0.0064
6     6  weekday           1575.0                 1.0714              0.0258
7     7  weekday           4830.0                 1.0743              0.0792
8     8  weekday           7376.0                 1.0765              0.1210
9     9  weekday           5495.0                 1.0743              0.0901


In [15]:
output_path = processed_dir / "agg_ridership_by_hour.csv"

agg_ridership_by_hour.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_ridership_by_hour))
print("Columns:", len(agg_ridership_by_hour.columns))

Saved: ..\data\processed\agg_ridership_by_hour.csv
Rows: 48
Columns: 5


In [16]:
agg_route_performance = (
    fact.groupby(
        [
            "route_id",
            "route_name",
            "mode",
            "district_x",
            "year_month"
        ]
    )
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        avg_load_factor=("load_factor", "mean"),
        route_length_km=("route_length_km", "first")
    )
    .reset_index()
)

agg_route_performance = agg_route_performance.rename(
    columns={"district_x": "district"}
)

agg_route_performance["boardings_per_km"] = (
    agg_route_performance["total_boardings"]
    / agg_route_performance["route_length_km"]
).round(4)

agg_route_performance["total_revenue"] = (
    agg_route_performance["total_revenue"].round(2)
)

agg_route_performance["avg_load_factor"] = (
    agg_route_performance["avg_load_factor"].round(4)
)

agg_route_performance = agg_route_performance[
    [
        "route_id",
        "route_name",
        "mode",
        "district",
        "year_month",
        "total_boardings",
        "total_revenue",
        "boardings_per_km",
        "avg_load_factor"
    ]
].sort_values(
    ["year_month", "total_boardings"],
    ascending=[True, False]
).reset_index(drop=True)

print(agg_route_performance.head())
print("Rows:", len(agg_route_performance))

  route_id             route_name   mode district year_month  total_boardings  \
0     R051       Daniel Lake Line    Bus   Centro    2024-01            284.0   
1     R032        Thomas Dam Line  Metro   Centro    2024-01            279.0   
2     R039  Carlson Mountain Line    Bus   Centro    2024-01            267.0   
3     R031     Wilkerson Row Line   Tram   Centro    2024-01            264.0   
4     R021      Martin Knoll Line   Tram  Westend    2024-01            263.0   

   total_revenue  boardings_per_km  avg_load_factor  
0         529.17           57.9592           0.0141  
1         478.92           11.8723           0.0018  
2         484.33           63.5714           0.0143  
3         485.74           15.6213           0.0054  
4         505.22           11.1441           0.0056  
Rows: 720


In [17]:
key_columns = [
    "route_id",
    "route_name",
    "mode",
    "district",
    "year_month"
]

assert not agg_route_performance[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [18]:
output_path = processed_dir / "agg_route_performance.csv"

agg_route_performance.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_route_performance))
print("Columns:", len(agg_route_performance.columns))

Saved: ..\data\processed\agg_route_performance.csv
Rows: 720
Columns: 9


In [19]:
agg_mode_summary = (
    fact.groupby(["mode", "year_month"])
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        route_count=("route_id", "nunique")
    )
    .reset_index()
)

agg_mode_summary["boardings_per_route"] = (
    agg_mode_summary["total_boardings"]
    / agg_mode_summary["route_count"]
).round(4)

agg_mode_summary["total_revenue"] = (
    agg_mode_summary["total_revenue"].round(2)
)

agg_mode_summary = (
    agg_mode_summary
    [
        [
            "mode",
            "year_month",
            "total_boardings",
            "total_revenue",
            "route_count",
            "boardings_per_route"
        ]
    ]
    .sort_values(["year_month", "total_boardings"], ascending=[True, False])
    .reset_index(drop=True)
)

print(agg_mode_summary)
print("Rows:", len(agg_mode_summary))

     mode year_month  total_boardings  total_revenue  route_count  \
0     Bus    2024-01           8021.0       14851.06           34   
1    Tram    2024-01           4599.0        8445.70           20   
2   Metro    2024-01           1380.0        2496.69            6   
3     Bus    2024-02           7395.0       13717.82           34   
4    Tram    2024-02           4510.0        8268.20           20   
5   Metro    2024-02           1223.0        2311.67            6   
6     Bus    2024-03           7889.0       14448.34           34   
7    Tram    2024-03           4754.0        8777.19           20   
8   Metro    2024-03           1379.0        2481.34            6   
9     Bus    2024-04           7652.0       14014.84           34   
10   Tram    2024-04           4464.0        8209.32           20   
11  Metro    2024-04           1298.0        2382.79            6   
12    Bus    2024-05           8062.0       14682.12           34   
13   Tram    2024-05           464

In [20]:
key_columns = ["mode", "year_month"]

assert not agg_mode_summary[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [21]:
output_path = processed_dir / "agg_mode_summary.csv"

agg_mode_summary.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_mode_summary))
print("Columns:", len(agg_mode_summary.columns))

Saved: ..\data\processed\agg_mode_summary.csv
Rows: 36
Columns: 6


In [22]:
agg_fare_type_mix = (
    fact.groupby(["fare_type", "year_month"])
    .agg(
        boardings=("passenger_count", "sum"),
        revenue=("revenue", "sum")
    )
    .reset_index()
)

agg_fare_type_mix["monthly_total"] = (
    agg_fare_type_mix
    .groupby("year_month")["boardings"]
    .transform("sum")
)

agg_fare_type_mix["pct_of_monthly_boardings"] = (
    agg_fare_type_mix["boardings"]
    / agg_fare_type_mix["monthly_total"]
).round(4)

agg_fare_type_mix["revenue"] = (
    agg_fare_type_mix["revenue"].round(2)
)

agg_fare_type_mix = (
    agg_fare_type_mix
    .drop(columns="monthly_total")
    .sort_values(["year_month", "boardings"], ascending=[True, False])
    .reset_index(drop=True)
)

print(agg_fare_type_mix)
print("Rows:", len(agg_fare_type_mix))

      fare_type year_month  boardings   revenue  pct_of_monthly_boardings
0         Adult    2024-01     7770.0  19419.30                    0.5550
1       Student    2024-01     2138.0   3213.49                    0.1527
2        Senior    2024-01     1695.0   2026.17                    0.1211
3   Pass Holder    2024-01     1339.0     78.12                    0.0956
4         Child    2024-01     1058.0   1056.37                    0.0756
5         Adult    2024-02     7357.0  18402.99                    0.5604
6       Student    2024-02     1988.0   2979.66                    0.1514
7        Senior    2024-02     1518.0   1822.32                    0.1156
8   Pass Holder    2024-02     1242.0     69.14                    0.0946
9         Child    2024-02     1023.0   1023.58                    0.0779
10        Adult    2024-03     7751.0  19369.05                    0.5528
11      Student    2024-03     2108.0   3152.63                    0.1503
12       Senior    2024-03     1685.0 

In [23]:
key_columns = ["fare_type", "year_month"]

assert not agg_fare_type_mix[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [24]:
output_path = processed_dir / "agg_fare_type_mix.csv"

agg_fare_type_mix.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_fare_type_mix))
print("Columns:", len(agg_fare_type_mix.columns))

Saved: ..\data\processed\agg_fare_type_mix.csv
Rows: 60
Columns: 5


In [25]:
agg_stop_activity = (
    fact.groupby(
        ["stop_id", "stop_name", "district_y", "zone", "has_shelter"]
    )
    .agg(
        total_boardings=("passenger_count", "sum")
    )
    .reset_index()
)

agg_stop_activity = agg_stop_activity.rename(
    columns={"district_y": "district"}
)

agg_stop_activity["rank_within_district"] = (
    agg_stop_activity
    .groupby("district")["total_boardings"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

agg_stop_activity = (
    agg_stop_activity[
        [
            "stop_id",
            "stop_name",
            "district",
            "zone",
            "total_boardings",
            "rank_within_district",
            "has_shelter"
        ]
    ]
    .sort_values(
        ["district", "rank_within_district", "stop_id"]
    )
    .reset_index(drop=True)
)

print(agg_stop_activity.head(10))
print("Rows:", len(agg_stop_activity))

  stop_id             stop_name district    zone  total_boardings  \
0   S0047     Joseph Coves Stop   Centro  Zone 3            237.0   
1   S0314      Davis Locks Stop   Centro  Zone 2            233.0   
2   S0394     Tucker Mills Stop   Centro  Zone 2            229.0   
3   S0103  Jennifer Lights Stop   Centro  Zone 1            228.0   
4   S0323     Brandon Camp Stop   Centro  Zone 1            228.0   
5   S0021      Lewis Locks Stop   Centro  Zone 3            226.0   
6   S0037      Emily Hills Stop   Centro  Zone 1            226.0   
7   S0391    Nicole Drives Stop   Centro  Zone 1            226.0   
8   S0215    Deborah Grove Stop   Centro  Zone 3            225.0   
9   S0014      James Plain Stop   Centro  Zone 3            221.0   

   rank_within_district  has_shelter  
0                     1        False  
1                     2         True  
2                     3        False  
3                     4        False  
4                     4        False  
5     

In [26]:
key_columns = [
    "stop_id",
    "stop_name",
    "district",
    "zone"
]

assert not agg_stop_activity[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [27]:
key_columns = [
    "stop_id",
    "stop_name",
    "district",
    "zone"
]

assert not agg_stop_activity[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [28]:
output_path = processed_dir / "agg_stop_activity.csv"

agg_stop_activity.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_stop_activity))
print("Columns:", len(agg_stop_activity.columns))

Saved: ..\data\processed\agg_stop_activity.csv
Rows: 400
Columns: 7


In [29]:
agg_vehicle_utilisation = (
    fact.groupby(
        ["vehicle_id", "vehicle_type", "depot", "vehicle_age_band"]
    )
    .agg(
        total_boardings=("passenger_count", "sum"),
        avg_load_factor=("load_factor", "mean"),
        active_days=("date", "nunique")
    )
    .reset_index()
)

agg_vehicle_utilisation["avg_load_factor"] = (
    agg_vehicle_utilisation["avg_load_factor"].round(4)
)

agg_vehicle_utilisation = (
    agg_vehicle_utilisation
    .sort_values("total_boardings", ascending=False)
    .reset_index(drop=True)
)

print(agg_vehicle_utilisation.head())
print("Rows:", len(agg_vehicle_utilisation))
print("Columns:", len(agg_vehicle_utilisation.columns))

  vehicle_id vehicle_type            depot vehicle_age_band  total_boardings  \
0      V0176         Tram      North Depot              20+            852.0   
1      V0116         Tram  Riverside Depot            11-20            814.0   
2      V0053         Tram    Central Depot              20+            801.0   
3      V0111         Tram      North Depot              0-5            788.0   
4      V0105         Tram      North Depot              20+            783.0   

   avg_load_factor  active_days  
0           0.0061          200  
1           0.0058          194  
2           0.0061          193  
3           0.0048          201  
4           0.0060          198  
Rows: 180
Columns: 7


In [30]:
key_columns = [
    "vehicle_id",
    "vehicle_type",
    "depot",
    "vehicle_age_band"
]

assert not agg_vehicle_utilisation[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [31]:
output_path = processed_dir / "agg_vehicle_utilisation.csv"

agg_vehicle_utilisation.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_vehicle_utilisation))
print("Columns:", len(agg_vehicle_utilisation.columns))

Saved: ..\data\processed\agg_vehicle_utilisation.csv
Rows: 180
Columns: 7


In [32]:
agg_peak_analysis = (
    fact.groupby(["time_period", "mode"])
    .agg(
        total_boardings=("passenger_count", "sum"),
        avg_load_factor=("load_factor", "mean")
    )
    .reset_index()
)

agg_peak_analysis["mode_total"] = (
    agg_peak_analysis
    .groupby("mode")["total_boardings"]
    .transform("sum")
)

agg_peak_analysis["pct_of_mode_total"] = (
    agg_peak_analysis["total_boardings"]
    / agg_peak_analysis["mode_total"]
).round(4)

agg_peak_analysis["avg_load_factor"] = (
    agg_peak_analysis["avg_load_factor"].round(4)
)

agg_peak_analysis = (
    agg_peak_analysis
    .drop(columns="mode_total")
    .sort_values(["mode", "total_boardings"], ascending=[True, False])
    .reset_index(drop=True)
)

print(agg_peak_analysis)
print("Rows:", len(agg_peak_analysis))
print("Columns:", len(agg_peak_analysis.columns))

   time_period   mode  total_boardings  avg_load_factor  pct_of_mode_total
0      PM Peak    Bus          15341.0           0.0134             0.3182
1      AM Peak    Bus          14041.0           0.0134             0.2912
2       Midday    Bus          11950.0           0.0135             0.2479
3      Evening    Bus           4806.0           0.0135             0.0997
4        Early    Bus           1650.0           0.0133             0.0342
5        Night    Bus            423.0           0.0126             0.0088
6      PM Peak  Metro           2620.0           0.0018             0.3145
7      AM Peak  Metro           2457.0           0.0018             0.2950
8       Midday  Metro           2085.0           0.0018             0.2503
9      Evening  Metro            828.0           0.0018             0.0994
10       Early  Metro            265.0           0.0017             0.0318
11       Night  Metro             75.0           0.0019             0.0090
12     PM Peak   Tram    

In [33]:
key_columns = ["time_period", "mode"]

assert not agg_peak_analysis[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [34]:
output_path = processed_dir / "agg_peak_analysis.csv"

agg_peak_analysis.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_peak_analysis))
print("Columns:", len(agg_peak_analysis.columns))

Saved: ..\data\processed\agg_peak_analysis.csv
Rows: 18
Columns: 5


In [35]:
agg_transfer_behaviour = (
    fact.groupby(["year_month", "mode"])
    .agg(
        transfer_count=("transfer_flag", "sum"),
        total_validations=("transfer_flag", "count")
    )
    .reset_index()
)

agg_transfer_behaviour["transfer_rate"] = (
    agg_transfer_behaviour["transfer_count"]
    / agg_transfer_behaviour["total_validations"]
).round(4)

agg_transfer_behaviour = (
    agg_transfer_behaviour
    .drop(columns="total_validations")
    .sort_values(["year_month", "mode"])
    .reset_index(drop=True)
)

print(agg_transfer_behaviour)
print("Rows:", len(agg_transfer_behaviour))
print("Columns:", len(agg_transfer_behaviour.columns))

   year_month   mode  transfer_count  transfer_rate
0     2024-01    Bus            2795         0.3765
1     2024-01  Metro             470         0.3707
2     2024-01   Tram            1665         0.3851
3     2024-02    Bus            2520         0.3670
4     2024-02  Metro             397         0.3489
5     2024-02   Tram            1598         0.3856
6     2024-03    Bus            2792         0.3830
7     2024-03  Metro             481         0.3802
8     2024-03   Tram            1685         0.3810
9     2024-04    Bus            2713         0.3806
10    2024-04  Metro             421         0.3514
11    2024-04   Tram            1630         0.3888
12    2024-05    Bus            2915         0.3883
13    2024-05  Metro             499         0.3883
14    2024-05   Tram            1611         0.3752
15    2024-06    Bus            2694         0.3897
16    2024-06  Metro             460         0.3725
17    2024-06   Tram            1602         0.3777
18    2024-0

In [36]:
key_columns = ["year_month", "mode"]

assert not agg_transfer_behaviour[key_columns].isna().any().any(), (
    "Null values found in key columns!"
)

print("Key-column null check passed.")

Key-column null check passed.


In [37]:
output_path = processed_dir / "agg_transfer_behaviour.csv"

agg_transfer_behaviour.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Rows:", len(agg_transfer_behaviour))
print("Columns:", len(agg_transfer_behaviour.columns))

Saved: ..\data\processed\agg_transfer_behaviour.csv
Rows: 36
Columns: 4


In [38]:
from datetime import datetime

output_names = [
    "agg_ridership_by_month",
    "agg_ridership_by_hour",
    "agg_route_performance",
    "agg_mode_summary",
    "agg_fare_type_mix",
    "agg_stop_activity",
    "agg_vehicle_utilisation",
    "agg_peak_analysis",
    "agg_transfer_behaviour"
]

manifest_rows = []

for name in output_names:
    file_path = processed_dir / f"{name}.csv"

    df = pd.read_csv(file_path)

    manifest_rows.append({
        "file_name": file_path.name,
        "row_count": len(df),
        "column_count": len(df.columns),
        "generated_at": datetime.now().isoformat(timespec="seconds")
    })

manifest = pd.DataFrame(manifest_rows)

manifest_path = processed_dir / "_manifest.csv"

manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8"
)

print(manifest)
print()
print("Manifest saved:", manifest_path)

                     file_name  row_count  column_count         generated_at
0   agg_ridership_by_month.csv         12             5  2026-08-03T14:18:22
1    agg_ridership_by_hour.csv         48             5  2026-08-03T14:18:22
2    agg_route_performance.csv        720             9  2026-08-03T14:18:22
3         agg_mode_summary.csv         36             6  2026-08-03T14:18:22
4        agg_fare_type_mix.csv         60             5  2026-08-03T14:18:22
5        agg_stop_activity.csv        400             7  2026-08-03T14:18:22
6  agg_vehicle_utilisation.csv        180             7  2026-08-03T14:18:22
7        agg_peak_analysis.csv         18             5  2026-08-03T14:18:22
8   agg_transfer_behaviour.csv         36             4  2026-08-03T14:18:22

Manifest saved: ..\data\processed\_manifest.csv


In [4]:
from pathlib import Path

processed_dir = Path("../data/processed")

expected_files = [
    "agg_ridership_by_month.csv",
    "agg_ridership_by_hour.csv",
    "agg_route_performance.csv",
    "agg_mode_summary.csv",
    "agg_fare_type_mix.csv",
    "agg_stop_activity.csv",
    "agg_vehicle_utilisation.csv",
    "agg_peak_analysis.csv",
    "agg_transfer_behaviour.csv",
    "_manifest.csv"
]

missing_files = [
    name for name in expected_files
    if not (processed_dir / name).exists()
]

assert not missing_files, f"Missing files: {missing_files}"

print("Final export check passed.")
print(f"All {len(expected_files)} files exist in:", processed_dir)

Final export check passed.
All 10 files exist in: ..\data\processed
